In [30]:
# ==============================================================================
# SCRIPT FINAL: VISUALISASI SPASIAL FASILITAS KAMPUS UNMUL
# Nama: M. Dedy Lesmana
# ==============================================================================

import pandas as pd
import geopandas as gpd                   # membaca file geospatial
import folium                             # membuat peta interaktif
from google.colab import drive
from folium.plugins import MarkerCluster  # membuat cluster marker

In [31]:
# --- TAHAPAN 1: PEMANGGILAN DATA  ---
drive.mount('/content/drive')
# Membaca dataset
path_data = '/content/drive/MyDrive/project/data_kampus.csv'
df = pd.read_csv(path_data, sep=';')
display(df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Nama_Gedung,Fungsi,Fakultas,Longitude,Latitude
0,Universitas_Mulawarman,Campus,Umum,117.153.998,-0.468451
1,Fakultas_Ekonomi_dan_Bisnis,Kuliah,FEB,117.154.421,-0.467609
2,Fakultas_MIPA,Kuliah,MIPA,117.153.081,-0.470041
3,Fakultas_Teknik,Kuliah,Teknik,117.158.993,-0.467574
4,Fakultas_Kedokteran,Kuliah,Kedokteran,117.155.989,-0.465643
5,Fakultas_Kehutanan,Kuliah,Kehutanan,117.153.465,-0.471166
6,Fakultas_Ilmu_Sosial_dan_Politik,Kuliah,FISIP,117.153.989,-0.469137
7,Perpustakaan_Universitas_Mulawarman,Fasilitas,Umum,117.155.114,-0.468322
8,Rektorat_Universitas_Mulawarman,Administrasi,Umum,117.154.165,-0.468405
9,GOR 27 September,Fasilitas,Umum,117.156.964,-0.468817


In [32]:
# Menampilkan nama kolom dataset
df.columns


Index(['Nama_Gedung', 'Fungsi', 'Fakultas', 'Longitude', 'Latitude'], dtype='object')

In [33]:
# Menampilkan jumlah baris dan kolom dataset
df.shape


(13, 5)

In [34]:
# Menampilkan ringkasan struktur data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Nama_Gedung  13 non-null     object 
 1   Fungsi       13 non-null     object 
 2   Fakultas     13 non-null     object 
 3   Longitude    13 non-null     object 
 4   Latitude     13 non-null     float64
dtypes: float64(1), object(4)
memory usage: 652.0+ bytes


In [35]:
# --- TAHAPAN 2: KALIBRASI & CLEANING KOORDINAT ---
def kustom_kalibrasi_longitude(nilai):
    nilai_str = str(nilai).strip()
    angka_bersih = nilai_str.replace('.', '')
    if len(angka_bersih) >= 3:
      return float(angka_bersih[:3] + '.' + angka_bersih[3:])
    return float(nilai_str)

# Terapkan kalibrasi presisi pada kolom Longitude dan Latitude
df['Longitude'] = df['Longitude'].apply(kustom_kalibrasi_longitude)
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')

print("--- Data Berhasil Dimuat dari Google Drive & Dikalibrasi ---")
print(df[['Nama_Gedung', 'Longitude', 'Latitude']].head())

--- Data Berhasil Dimuat dari Google Drive & Dikalibrasi ---
                   Nama_Gedung   Longitude  Latitude
0       Universitas_Mulawarman  117.153998 -0.468451
1  Fakultas_Ekonomi_dan_Bisnis  117.154421 -0.467609
2                Fakultas_MIPA  117.153081 -0.470041
3              Fakultas_Teknik  117.158993 -0.467574
4          Fakultas_Kedokteran  117.155989 -0.465643


In [40]:
# --- TAHAPAN 4: PEMBUATAN GEODATAFRAME ---
# Mentransformasi koordinat X (Longitude) dan Y (Latitude) menjadi objek spasial Point
geometry = gpd.points_from_xy(df['Longitude'], df['Latitude'])

# Membentuk GeoDataFrame dengan sistem proyeksi geografis standar WGS84 (EPSG:4326)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

In [41]:
# --- TAHAPAN 5: VISUALISASI INTERAKTIF (FOLIUM) ---
# Menentukan titik tengah kamera berdasarkan rata-rata koordinat gedung
center_lat = gdf['Latitude'].mean()
center_lon = gdf['Longitude'].mean()

# Inisialisasi kanvas peta dasar menggunakan OpenStreetMap
peta_kampus = folium.Map(location=[center_lat, center_lon], zoom_start=16, tiles='OpenStreetMap')

# Aturan pewarnaan penanda (marker) berdasarkan fungsi gedung
def tentukan_warna_marker(fungsi_gedung):
    if fungsi_gedung == 'Campus': return 'darkblue'
    elif fungsi_gedung == 'Kuliah': return 'red'
    elif fungsi_gedung == 'Fasilitas': return 'green'
    elif fungsi_gedung == 'Administrasi': return 'orange'
    return 'gray'

# Melakukan perulangan (looping) untuk menempelkan marker ke peta dasar
for index, row in gdf.iterrows():
# Merancang struktur Popup HTML berbentuk tabel interaktif yang estetis
    html_popup = f"""
    <div style='font-family: Arial, sans-serif; width: 200px;'>
        <h4 style='margin: 0 0 5px 0; color: #2C3E50;'><b>{row['Nama_Gedung'].replace('_', ' ')}</b></h4>
        <table style='width: 100%; border-collapse: collapse; font-size: 12px;'>
            <tr><td style='padding: 3px 0; width: 70px;'><b>Fungsi:</b></td><td>{row['Fungsi']}</td></tr>
            <tr><td style='padding: 3px 0;'><b>Fakultas:</b></td><td>{row['Fakultas']}</td></tr>
        </table>
    </div>
    """

# Menambahkan marker objek ke peta
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(html_popup, max_width=250),
        tooltip=row['Nama_Gedung'].replace('_', ' '),
        icon=folium.Icon(color=tentukan_warna_marker(row['Fungsi']), icon='info-sign')
    ).add_to(peta_kampus)

# Fitur fit_bounds agar kamera otomatis melakukan auto-zoom membidik semua marker
daftar_titik = gdf[['Latitude', 'Longitude']].dropna().values.tolist()
peta_kampus.fit_bounds(daftar_titik)

# Menyimpan peta
peta_kampus.save('peta_interaktif_kampus_unmul.html')
print("\n[SUKSES] File 'peta_interaktif_kampus_unmul.html' berhasil diexport!")



[SUKSES] File 'peta_interaktif_kampus_unmul.html' berhasil diexport!


In [39]:
# Menampilkan peta secara langsung di lembar kerja Google Colab
peta_kampus